### Raw Data Cleaning

In [1]:
import pandas as pd

In [2]:
locations_df = pd.read_csv("../data/locations.csv")
providers_df = pd.read_csv("../data/providers.csv")
payers_df = pd.read_csv("../data/payers.csv")
patients_df = pd.read_csv("../data/patients.csv")
calls_df = pd.read_csv("../data/calls.csv")
appointments_df = pd.read_csv("../data/appointments.csv")

for name, df in [("locations", locations_df), ("providers", providers_df),
                  ("payers", payers_df), ("patients", patients_df),
                  ("calls", calls_df), ("appointments", appointments_df)]:
    print(name, df.shape)

locations (7, 4)
providers (27, 5)
payers (7, 3)
patients (26554, 4)
calls (47300, 6)
appointments (200417, 12)


In [3]:
def strip_text_columns(df):
    text_cols = df.select_dtypes(include=['object', 'string']).columns
    for col in text_cols:
        df[col] = df[col].str.strip()
        
    return df

locations_df = strip_text_columns(locations_df)
payers_df = strip_text_columns(payers_df)
patients_df = strip_text_columns(patients_df)
calls_df = strip_text_columns(calls_df)
providers_df = strip_text_columns(providers_df)
appointments_df = strip_text_columns(appointments_df)

def strip_column_titles(df):
    df.columns = df.columns.str.strip().str.lower().str.replace(' ', '_')

    return df

locations_df = strip_column_titles(locations_df)
payers_df = strip_column_titles(payers_df)
patients_df = strip_column_titles(patients_df)
calls_df = strip_column_titles(calls_df)
providers_df = strip_column_titles(providers_df)
appointments_df = strip_column_titles(appointments_df)

### Appointments

In [4]:
appointments_df.head()

,appointment_id,date,booked_date,provider_id,location_id,patient_id,payer_id,appointment_type,is_new_patient,status,revenue,rvu
0,A1,2022-07-20,2022-07-20,P2,L1,PT1,PY1,Follow-up,False,Completed,125.50,0.81
1,A2,2022-07-20,2022-07-19,P2,L1,PT1,PY1,Follow-up,False,Completed,75.40,0.83
2,A3,2022-07-20,2022-07-02,P2,L1,PT2,PY3,New Patient Consult,True,Completed,258.34,2.47
3,A4,2022-07-20,2022-07-03,P2,L1,PT2,PY3,Follow-up,False,Completed,163.69,1.15
4,A5,2022-07-20,2022-07-06,P2,L1,PT3,PY5,New Patient Consult,True,Completed,297.58,1.97


In [5]:
appointments_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 200417 entries, 0 to 200416
Data columns (total 12 columns):
 #   Column            Non-Null Count   Dtype  
---  ------            --------------   -----  
 0   appointment_id    200417 non-null  object 
 1   date              200417 non-null  object 
 2   booked_date       200417 non-null  object 
 3   provider_id       200417 non-null  object 
 4   location_id       200417 non-null  object 
 5   patient_id        200417 non-null  object 
 6   payer_id          200417 non-null  object 
 7   appointment_type  200417 non-null  object 
 8   is_new_patient    200417 non-null  bool   
 9   status            200417 non-null  object 
 10  revenue           198758 non-null  float64
 11  rvu               200417 non-null  float64
dtypes: bool(1), float64(2), object(9)
memory usage: 17.0+ MB


In [6]:
# Convert dates to datetime format
appointments_df['date'] = pd.to_datetime(appointments_df['date'], errors='coerce', format='%Y-%m-%d')
appointments_df['booked_date'] = pd.to_datetime(appointments_df['booked_date'], errors='coerce', format='%Y-%m-%d')

In [7]:
# Clean column names by stripping whitespace and converting to lowercase
# appointments_df.columns = appointments_df.columns.str.strip().str.lower().str.replace(' ', '_').str.replace('(', '').str.replace(')', '')
# appointments_df.columns.tolist()

# Redundant now, swap to bulk processing for all dataframes

In [8]:
appointments_df.isna().sum()

appointment_id         0
date                   0
booked_date            0
provider_id            0
location_id            0
patient_id             0
payer_id               0
appointment_type       0
is_new_patient         0
status                 0
revenue             1659
rvu                    0
dtype: int64

In [9]:
# Find rows where revenue is null and status is Completed
# Meaning that the appointment was completed but revenue was not posted yet, might require followup if lost
blank_revenue_df = appointments_df[(appointments_df['revenue'].isna()) & (appointments_df['status'] == 'Completed')]
blank_revenue_df.shape

(1659, 12)

In [10]:
# No rows where revenue is 0 and status is Completed, so no need to follow up on those
appointments_df[(appointments_df['revenue'] == 0) & (appointments_df['status'] == 'Completed')]

,appointment_id,date,booked_date,provider_id,location_id,patient_id,payer_id,appointment_type,is_new_patient,status,revenue,rvu


In [11]:
# Impossible dates, turn them into NaT but keep records for safekeeping rather than dropping completely
bad_dates = appointments_df['booked_date'] > appointments_df['date']
appointments_df.loc[bad_dates, 'booked_date'] = pd.NaT

In [12]:
appointments_df['booked_date'].value_counts()

booked_date
2026-07-01    294
2026-07-28    277
2026-07-13    276
2026-07-08    274
2026-08-04    270
             ... 
2022-07-06      1
2022-07-13      1
2022-07-02      1
2022-07-12      1
2022-07-07      1
Name: count, Length: 1523, dtype: int64

In [13]:
# Primary key validation
appointments_df['appointment_id'].nunique() == len(appointments_df)

False

In [14]:
# Drop duplicate rows based on the 'appointment_id' column
appointments_df.drop_duplicates(subset=['appointment_id'], keep='first', inplace=True)
appointments_df.shape # 200417 -> 199420

(199420, 12)

In [15]:
# Fine now
appointments_df['appointment_id'].nunique() == len(appointments_df)

True

In [16]:
# Provider validation, find appointments with provider_id that doesn't exist in providers_df
set(providers_df['provider_id'])
provider_error_appointments_df = appointments_df[~appointments_df['provider_id'].isin(set(providers_df['provider_id']))]
provider_error_appointments_df

,appointment_id,date,booked_date,provider_id,location_id,patient_id,payer_id,appointment_type,is_new_patient,status,revenue,rvu
462,A463,2022-09-13,2022-09-05,P99,L5,PT9,PY1,Follow-up,False,Completed,129.52,1.24
1469,A1470,2022-12-07,2022-11-30,P99,L1,PT42,PY3,Post-Op Check,False,Completed,109.75,0.82
1557,A1558,2022-12-12,2022-12-05,P99,L6,PT204,PY3,Follow-up,False,Cancelled,0.00,0.00
2223,A2224,2023-01-16,2023-01-04,P99,L1,PT174,PY1,Post-Op Check,False,Completed,104.82,0.78
2274,A2275,2023-01-18,2023-01-08,P99,L1,PT148,PY4,Follow-up,False,Completed,143.17,0.93
...,...,...,...,...,...,...,...,...,...,...,...,...
198223,A198224,2026-09-01,2026-08-15,P99,L3,PT3463,PY3,Follow-up,False,Completed,214.23,1.26
198340,A198341,2026-09-01,2026-08-27,P99,L7,PT14021,PY1,Physical Therapy,False,Completed,74.14,1.29
198556,A198557,2026-09-02,2026-08-19,P99,L5,PT2284,PY3,Follow-up,False,Completed,123.02,0.97
199068,A199069,2026-09-04,2026-08-31,P99,L3,PT9493,PY3,Follow-up,False,Completed,157.21,0.69


In [17]:
# P99 doesn't exist in providers_df, but it does exist in appointments_df. This is likely a data quality issue that needs to be addressed
set(appointments_df['provider_id'])

{'P1',
 'P10',
 'P11',
 'P12',
 'P13',
 'P14',
 'P15',
 'P16',
 'P17',
 'P18',
 'P19',
 'P2',
 'P20',
 'P21',
 'P22',
 'P23',
 'P24',
 'P25',
 'P26',
 'P27',
 'P3',
 'P4',
 'P5',
 'P6',
 'P7',
 'P8',
 'P9',
 'P99'}

In [18]:
# Some provider_ids in appointments_df do not exist in providers_df. Replace those with 'UNK' to indicate unknown provider
invalid_provider_mask = ~appointments_df['provider_id'].isin(set(providers_df['provider_id']))
appointments_df.loc[invalid_provider_mask, 'provider_id'] = 'UNK'

appointments_df[appointments_df['provider_id'] == 'UNK']

# Need to add a row to providers_df for the unknown provider so that the join works correctly later on
unknown_provider = pd.DataFrame([{
    'provider_id': 'UNK', 'provider_name': 'Unknown Provider',
    'specialty': 'Unknown', 'primary_location_id': None, 'hire_date': pd.NaT
}])
providers_df = pd.concat([providers_df, unknown_provider], ignore_index=True)
providers_df

,provider_id,provider_name,specialty,primary_location_id,hire_date
0,P1,Dr. James Nguyen,Orthopedic Surgery,L1,2022-11-13
1,P2,Dr. Maria Patel,Sports Medicine,L1,2022-07-20
2,P3,Dr. Robert Garcia,Spine Surgery,L1,2022-10-14
3,P4,Dr. David Rossi,Physical Medicine & Rehab,L1,2023-02-08
4,P5,Dr. Susan Chen,Orthopedic Surgery,L2,2022-12-24
5,P6,Dr. Michael Alvarez,Sports Medicine,L2,2023-10-17
6,P7,"Karen Bennett, PT",Physical Therapy,L2,2024-02-23
7,P8,Dr. Thomas Walsh,Pain Management,L2,2024-01-12
8,P9,Dr. Nancy Okafor,Orthopedic Surgery,L3,2023-02-09
9,P10,Dr. George Tanaka,Hand & Upper Extremity,L3,2023-05-07


In [19]:
appointments_df[~appointments_df['location_id'].isin(set(locations_df['location_id']))]

,appointment_id,date,booked_date,provider_id,location_id,patient_id,payer_id,appointment_type,is_new_patient,status,revenue,rvu


In [20]:
appointments_df[~appointments_df['patient_id'].isin(set(patients_df['patient_id']))]

,appointment_id,date,booked_date,provider_id,location_id,patient_id,payer_id,appointment_type,is_new_patient,status,revenue,rvu


In [21]:
appointments_df[~appointments_df['payer_id'].isin(set(payers_df['payer_id']))]

,appointment_id,date,booked_date,provider_id,location_id,patient_id,payer_id,appointment_type,is_new_patient,status,revenue,rvu


In [22]:
appointments_df.shape

(199420, 12)

In [23]:
# Check for negative revenue values
appointments_df[appointments_df['revenue'] < 0]['revenue'].sum()

np.float64(0.0)

In [24]:
# Check for negative rvu values
appointments_df[appointments_df['rvu'] < 0]['rvu'].sum()

np.float64(0.0)

In [25]:
appointments_df['status'].value_counts()

status
Completed    165110
No-Show       24168
Cancelled     10142
Name: count, dtype: int64

In [26]:
# Revenue should only ever be present on Completed appointments
# These are basically appointments that never happened (canceled, no-show, etc.) so the revenue are just 0.00
appointments_df[(appointments_df['status'] != 'Completed') & (appointments_df['revenue'].notna())]['revenue'].unique()

array([0.])

In [27]:
appointments_df['appointment_type'].value_counts()

appointment_type
Follow-up              77276
Post-Op Check          35520
Physical Therapy       35435
New Patient Consult    26553
Injection/Procedure    24636
Name: count, dtype: int64

In [28]:
# Check if the number of new patient consultations matches the number of new patients
len(appointments_df[appointments_df['appointment_type'] == 'New Patient Consult']) == len(appointments_df[appointments_df['is_new_patient'] == True])

True

In [29]:
# Checks if there's any non-completed appointments that have revenue greater than 0, which should not happen
appointments_df[(appointments_df['status'] != 'Completed') & (appointments_df['revenue'] > 0)]

,appointment_id,date,booked_date,provider_id,location_id,patient_id,payer_id,appointment_type,is_new_patient,status,revenue,rvu


In [30]:
# Makes sure that the revenue values are reasonable
appointments_df['revenue'].describe()

count    197771.000000
mean        146.979828
std         144.977798
min           0.000000
25%          73.590000
50%         110.770000
75%         164.700000
max         926.430000
Name: revenue, dtype: float64

In [31]:
appointments_df['rvu'].describe()

count    199420.000000
mean          1.204178
std           1.189213
min           0.000000
25%           0.610000
50%           0.850000
75%           1.250000
max           6.100000
Name: rvu, dtype: float64

In [32]:
# 197771 (Revenue count) + 1649 = 199420 (RVU count), which is the total number of appointments in the dataframe.
# 1759 appointments are waiting on the calculated revenue, but they should all have rvu which works out in this case.
appointments_df['revenue'].isna().sum()

np.int64(1649)

### Patients

In [33]:
patients_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 26554 entries, 0 to 26553
Data columns (total 4 columns):
 #   Column            Non-Null Count  Dtype 
---  ------            --------------  ----- 
 0   patient_id        26554 non-null  object
 1   first_visit_date  26554 non-null  object
 2   referral_source   26554 non-null  object
 3   payer_id          26554 non-null  object
dtypes: object(4)
memory usage: 829.9+ KB


In [34]:
# Primary key validation
patients_df['patient_id'].nunique() == len(patients_df)

True

In [35]:
patients_df['first_visit_date'] = pd.to_datetime(patients_df['first_visit_date'], errors='coerce', format='%Y-%m-%d')

In [36]:
patients_df['first_visit_date'].isna().sum()

np.int64(0)

In [37]:
patients_df['first_visit_date'].value_counts()

first_visit_date
2026-08-31    59
2026-08-28    57
2026-08-10    56
2026-08-14    55
2026-04-03    53
              ..
2022-08-31     1
2022-07-26     1
2022-09-10     1
2024-03-30     1
2022-09-17     1
Name: count, Length: 1265, dtype: int64

In [38]:
patients_df.head()

,patient_id,first_visit_date,referral_source,payer_id
0,PT1,2022-07-20,physician referral,PY1
1,PT2,2022-07-20,Physician Referral,PY3
2,PT3,2022-07-20,Physician Referral,PY5
3,PT4,2022-07-21,Self,PY3
4,PT5,2022-07-22,Friend/Family,PY4


In [39]:
# Inaccurate casing causing values to be counted separately when they are actually the same
patients_df['referral_source'].value_counts()

referral_source
Physician Referral     10161
Self                    5125
Insurance Directory     3782
Online Search           3680
Friend/Family           2648
Physician referral       443
physician referral       439
self                     142
SELF                     134
Name: count, dtype: int64

In [40]:
# Clean and check
patients_df['referral_source'] = patients_df['referral_source'].str.strip().str.title()
patients_df['referral_source'].value_counts()

referral_source
Physician Referral     11043
Self                    5401
Insurance Directory     3782
Online Search           3680
Friend/Family           2648
Name: count, dtype: int64

In [41]:
# No errors
invalid_payer_mask = ~patients_df['payer_id'].isin(set(payers_df['payer_id']))
patients_df[invalid_payer_mask]

,patient_id,first_visit_date,referral_source,payer_id


In [42]:
# Nothing dropped
patients_df.drop_duplicates()

,patient_id,first_visit_date,referral_source,payer_id
0,PT1,2022-07-20,Physician Referral,PY1
1,PT2,2022-07-20,Physician Referral,PY3
2,PT3,2022-07-20,Physician Referral,PY5
3,PT4,2022-07-21,Self,PY3
4,PT5,2022-07-22,Friend/Family,PY4
...,...,...,...,...
26549,PT26550,2026-09-05,Self,PY5
26550,PT26551,2026-09-05,Physician Referral,PY7
26551,PT26552,2026-09-05,Self,PY3
26552,PT26553,2026-09-05,Physician Referral,PY7


### Payers

In [43]:
payers_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7 entries, 0 to 6
Data columns (total 3 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   payer_id    7 non-null      object
 1   payer_name  7 non-null      object
 2   payer_type  7 non-null      object
dtypes: object(3)
memory usage: 300.0+ bytes


In [44]:
payers_df

,payer_id,payer_name,payer_type
0,PY1,Medicare,Medicare
1,PY2,Medi-Cal,Medicaid
2,PY3,Blue Cross,Commercial
3,PY4,Aetna,Commercial
4,PY5,Humana,Commercial
5,PY6,Self-Pay,Self-Pay
6,PY7,UnitedHealth,Commercial


In [45]:
# Primary key validation
payers_df['payer_id'].nunique() == len(payers_df)

True

In [46]:
payers_df['payer_type'].value_counts()

payer_type
Commercial    4
Medicare      1
Medicaid      1
Self-Pay      1
Name: count, dtype: int64

In [47]:
# Mostly insurance, but there are some self-pay patients as well
# Name and type matches, ex. Medicare with Medicare
payers_df[['payer_name', 'payer_type']]

,payer_name,payer_type
0,Medicare,Medicare
1,Medi-Cal,Medicaid
2,Blue Cross,Commercial
3,Aetna,Commercial
4,Humana,Commercial
5,Self-Pay,Self-Pay
6,UnitedHealth,Commercial


### Providers

In [48]:
providers_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 28 entries, 0 to 27
Data columns (total 5 columns):
 #   Column               Non-Null Count  Dtype 
---  ------               --------------  ----- 
 0   provider_id          28 non-null     object
 1   provider_name        28 non-null     object
 2   specialty            28 non-null     object
 3   primary_location_id  27 non-null     object
 4   hire_date            27 non-null     object
dtypes: object(5)
memory usage: 1.2+ KB


In [49]:
providers_df

,provider_id,provider_name,specialty,primary_location_id,hire_date
0,P1,Dr. James Nguyen,Orthopedic Surgery,L1,2022-11-13
1,P2,Dr. Maria Patel,Sports Medicine,L1,2022-07-20
2,P3,Dr. Robert Garcia,Spine Surgery,L1,2022-10-14
3,P4,Dr. David Rossi,Physical Medicine & Rehab,L1,2023-02-08
4,P5,Dr. Susan Chen,Orthopedic Surgery,L2,2022-12-24
5,P6,Dr. Michael Alvarez,Sports Medicine,L2,2023-10-17
6,P7,"Karen Bennett, PT",Physical Therapy,L2,2024-02-23
7,P8,Dr. Thomas Walsh,Pain Management,L2,2024-01-12
8,P9,Dr. Nancy Okafor,Orthopedic Surgery,L3,2023-02-09
9,P10,Dr. George Tanaka,Hand & Upper Extremity,L3,2023-05-07


In [50]:
providers_df['hire_date'] = pd.to_datetime(providers_df['hire_date'], errors='coerce', format='%Y-%m-%d')

In [51]:
providers_df['hire_date'].isna().sum()

np.int64(1)

In [52]:
# Validate that all primary_location_id values in providers_df exist in locations_df
invalid_location_mask = ~providers_df['primary_location_id'].isin(set(locations_df['location_id']))
providers_df[invalid_location_mask]

,provider_id,provider_name,specialty,primary_location_id,hire_date
27,UNK,Unknown Provider,Unknown,None,NaT


In [53]:
# Primary key validation
providers_df['provider_id'].nunique() == len(providers_df)

True

In [54]:
providers_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 28 entries, 0 to 27
Data columns (total 5 columns):
 #   Column               Non-Null Count  Dtype         
---  ------               --------------  -----         
 0   provider_id          28 non-null     object        
 1   provider_name        28 non-null     object        
 2   specialty            28 non-null     object        
 3   primary_location_id  27 non-null     object        
 4   hire_date            27 non-null     datetime64[ns]
dtypes: datetime64[ns](1), object(4)
memory usage: 1.2+ KB


### Locations

In [55]:
locations_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7 entries, 0 to 6
Data columns (total 4 columns):
 #   Column         Non-Null Count  Dtype 
---  ------         --------------  ----- 
 0   location_id    7 non-null      object
 1   location_name  7 non-null      object
 2   city           7 non-null      object
 3   state          7 non-null      object
dtypes: object(4)
memory usage: 356.0+ bytes


In [56]:
locations_df.head()

,location_id,location_name,city,state
0,L1,Santa Clarita Office,Santa Clarita,CA
1,L2,Valencia Office,Valencia,CA
2,L3,Burbank Office,Burbank,CA
3,L4,Glendale Office,Glendale,CA
4,L5,Pasadena Office,Pasadena,CA


In [57]:
# Primary key validation
locations_df['location_id'].nunique() == len(locations_df)

True

In [58]:
# Small sample of locations_df, but all states should be CA. This is just a check for good practice.
len(locations_df[locations_df['state'] == 'CA']) == len(locations_df)

True

In [59]:
locations_df['city'].value_counts()

city
Santa Clarita    1
Valencia         1
Burbank          1
Glendale         1
Pasadena         1
Thousand Oaks    1
Northridge       1
Name: count, dtype: int64

In [60]:
locations_df['location_name'].value_counts()

location_name
Santa Clarita Office    1
Valencia Office         1
Burbank Office          1
Glendale Office         1
Pasadena Office         1
Thousand Oaks Office    1
Northridge Office       1
Name: count, dtype: int64

### Calls

In [61]:
calls_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 47300 entries, 0 to 47299
Data columns (total 6 columns):
 #   Column           Non-Null Count  Dtype 
---  ------           --------------  ----- 
 0   call_id          47300 non-null  object
 1   date             47300 non-null  object
 2   location_id      47300 non-null  object
 3   call_type        47300 non-null  object
 4   outcome          47300 non-null  object
 5   handle_time_sec  47300 non-null  int64 
dtypes: int64(1), object(5)
memory usage: 2.2+ MB


In [62]:
calls_df.head()

,call_id,date,location_id,call_type,outcome,handle_time_sec
0,C1,2022-07-01,L1,New Patient Inquiry,Not Booked,209
1,C2,2022-07-01,L1,Reschedule,Booked,103
2,C3,2022-07-01,L1,Billing Question,Info Only,198
3,C4,2022-07-01,L1,Billing Question,Info Only,396
4,C5,2022-07-01,L2,New Patient Inquiry,Booked,255


In [63]:
calls_df['date'] = pd.to_datetime(calls_df['date'], errors='coerce', format='%Y-%m-%d')

In [64]:
# Validate that all location_id values in calls_df exist in locations_df
invalid_location_mask = ~calls_df['location_id'].isin(set(locations_df['location_id']))
calls_df[invalid_location_mask]

,call_id,date,location_id,call_type,outcome,handle_time_sec


In [65]:
# Primary key validation
calls_df['call_id'].nunique() == len(calls_df)

True

In [66]:
calls_df['call_type'].value_counts()

call_type
New Patient Inquiry    16058
Reschedule             14590
General Inquiry         9531
Billing Question        7121
Name: count, dtype: int64

In [67]:
calls_df['outcome'].value_counts()

outcome
Booked        22343
Info Only     14791
Not Booked    10166
Name: count, dtype: int64

In [68]:
# Make sure that values makes sense, ex. handle_time_sec should be positive and not too high
calls_df['handle_time_sec'].describe()

count    47300.000000
mean       275.204123
std        135.949194
min         40.000000
25%        158.000000
50%        275.000000
75%        393.000000
max        510.000000
Name: handle_time_sec, dtype: float64